# DATA-445: In-Class Assignment 12

## Overview
Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

## Data Attributes

### Demographic:
- **Sex**: male or female (Nominal)
- **Age**: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

### Behavioral:
- **Current Smoker**: whether or not the patient is a current smoker (Nominal)
- **Cigs Per Day**: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette)

### Medical (history):
- **BP Meds**: whether or not the patient was on blood pressure medication (Nominal)
- **Prevalent Stroke**: whether or not the patient had previously had a stroke (Nominal)
- **Prevalent Hyp**: whether or not the patient was hypertensive (Nominal)
- **Diabetes**: whether or not the patient had diabetes (Nominal)

### Medical (current):
- **Tot Chol**: total cholesterol level (Continuous)
- **Sys BP**: systolic blood pressure (Continuous)
- **Dia BP**: diastolic blood pressure (Continuous)
- **BMI**: Body Mass Index (Continuous)
- **Heart Rate**: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values)
- **Glucose**: glucose level (Continuous)

### Target Variable:
- **TenYearCHD**: 10 year risk of coronary heart disease CHD (binary: "1" means "Yes", "0" means "No")

## Exercise 1: Data Loading and Preprocessing (4 points)

Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [4]:
import pandas as pd

heart = pd.read_csv("framingham.csv")
heart.head()

heart = heart.dropna().reset_index(drop=True)

heart.isnull().sum()

male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
TenYearCHD         0
dtype: int64

## Exercise 2: Define Features and Cross-Validation Strategy (4 points)

Define `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the input variables, and `TenYearCHD` is the target variable. Define a 5-fold cross-validation strategy.

In [5]:
from sklearn.model_selection import StratifiedKFold

X = heart[["age","currentSmoker","totChol","BMI","heartRate"]]
Y = heart["TenYearCHD"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Exercise 3: AdaBoostClassifier Evaluation (8 points)

Train and evaluate an `AdaBoostClassifier` model (with `n_estimators = 500`, `max_depth = 3`, and `learning_rate = 0.01`) over the 5-fold cross-validation strategy defined in Exercise 2. Using 10% as cutoff value, report the average recall over the 5-folds.

In [8]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import recall_score
from sklearn.tree import DecisionTreeClassifier

ada_recalls = []

for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]

    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=500,
        learning_rate=0.01,
        random_state=42
    )
    ada.fit(X_train, Y_train)
    Y_pred = ada.predict_proba(X_test)[:, 1]
    Y_pred = (Y_pred >= 0.1).astype(int)
    
    score = recall_score(Y_test, Y_pred)
    ada_recalls.append(score)
    print(f"Fold {i+1} recall: {score}")

print(f"Average recall: {sum(ada_recalls)/len(ada_recalls)}")

Fold 1 recall: 1.0
Fold 2 recall: 1.0
Fold 3 recall: 1.0
Fold 4 recall: 1.0
Fold 5 recall: 1.0
Average recall: 1.0


## Exercise 4: RandomForestClassifier Evaluation (8 points)

Train and evaluate a `RandomForestClassifier` model (with `n_estimators = 500` and `max_depth = 3`) over the 5-fold cross-validation strategy defined in Exercise 2. Using 10% as cutoff value, report the average recall over the 5-folds.

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]
    
    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=3,
        random_state=42,
        n_jobs=-1,
    )
    
    rf.fit(X_train, Y_train)
    Y_pred = rf.predict_proba(X_test)[:, 1]
    Y_pred = (Y_pred >= 0.1).astype(int)
    
    score = recall_score(Y_test, Y_pred)
    rf_recalls.append(score)
    print(f"Fold {i+1} recall: {score}")

print(f"Average recall: {sum(rf_recalls)/len(rf_recalls)}")

Fold 1 recall: 0.7678571428571429
Fold 2 recall: 0.8468468468468469
Fold 3 recall: 0.9369369369369369
Fold 4 recall: 0.9279279279279279
Fold 5 recall: 0.8571428571428571
Average recall: 0.8673423423423424


## Exercise 5: ExtraTreesClassifier Evaluation (8 points)

Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators = 500` and `max_depth = 3`) over the 5-fold cross-validation strategy defined in Exercise 2. Using 10% as cutoff value, report the average recall over the 5-folds.

In [10]:
from sklearn.ensemble import ExtraTreesClassifier

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]

    et = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=3,
        random_state=42,
        n_jobs=-1,
    )

    et.fit(X_train, Y_train)
    Y_pred = et.predict_proba(X_test)[:, 1]
    Y_pred = (Y_pred >= 0.1).astype(int)

    score = recall_score(Y_test, Y_pred)
    et_recalls.append(score)
    print(f"Fold {i+1} recall: {score}")

print(f"Average recall: {sum(et_recalls)/len(et_recalls)}")

Fold 1 recall: 0.9821428571428571
Fold 2 recall: 0.9819819819819819
Fold 3 recall: 1.0
Fold 4 recall: 1.0
Fold 5 recall: 0.9732142857142857
Average recall: 0.9874678249678249


## Exercise 6: Model Comparison and Selection (3 points)

Using the results from Exercises 3, 4, and 5, what model would you use to predict `TenYearCHD` based on the recall? Justify your choice.

based on my results, i would use the adaboost to predict TenYearCHD because the average recall is higher.